# Example analysis using the genulens CLI

This notebook provides practical event-analysis examples using the `genulens` command-line executable.

Most commands are adapted from [Usage.pdf](https://github.com/nkoshimoto/genulens/blob/main/Usage.pdf), but this notebook presents them on an event-by-event basis rather than option-by-option. See section 3 of [Usage.pdf](https://github.com/nkoshimoto/genulens/blob/main/Usage.pdf) for details of each parameter used below.

Run this notebook from either the repository root or the `examples/` directory after building genulens. The helper below looks for `build/genulens` first, then falls back to `./genulens` in the repository root.


In [ ]:
from pathlib import Path
import os
import re
import shlex
import subprocess
from subprocess import PIPE

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Allow this notebook to be launched either from the repository root or examples/.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "examples" else cwd
output_dir = cwd

GENULENS_EXE = repo_root / "build" / "genulens"
if not GENULENS_EXE.exists():
    GENULENS_EXE = repo_root / "genulens"
if not GENULENS_EXE.exists():
    raise FileNotFoundError(
        "Could not find genulens executable. Build the project first, e.g. `cmake --build build`."
    )

GENULENS_EXE


In [ ]:
# To run genulens through Python.

def get_header(argv):
    VERBOSITY, BINARY = 0, 0
    args = shlex.split(argv)
    for i, arg in enumerate(args):
        if arg == "VERBOSITY":
            VERBOSITY = int(args[i + 1])
        if arg == "BINARY":
            BINARY = int(args[i + 1])

    if VERBOSITY == 3:
        header = "wtj M_L D_L D_S t_E theta_E pi_E pi_EN pi_EE mu_rel mu_Sl mu_Sb I_L K_L iS iL fREM".split()
    elif VERBOSITY == 2:
        header = "wtj t_E theta_E pi_EN pi_EE D_S mu_Sl mu_Sb iS iL fREM".split()
    else:
        header = None
    if BINARY == 1 and VERBOSITY > 1:
        header += "q21 M2 aL aLpmin u0 BL".split()
    return header


def run_genulens(file, argv, ranseed=False, grep=False):
    args = shlex.split(argv)
    if ranseed:
        args += ["seed", str(np.random.randint(1, 10**10))]

    output_path = output_dir / file
    printable = " ".join([shlex.quote(str(GENULENS_EXE)), *map(shlex.quote, args)])
    if grep:
        printable += f" | grep {grep}"
    printable += f" > {shlex.quote(str(output_path))}"
    print("# Conducted command (you can do this directly in your command line too):")
    print("\n", printable, "\n")

    proc = subprocess.run(
        [str(GENULENS_EXE), *args],
        cwd=repo_root,
        stdout=PIPE,
        stderr=PIPE,
        text=True,
        check=True,
    )
    stdout = proc.stdout
    if grep:
        pattern = re.compile(grep)
        stdout = "\n".join(line for line in stdout.splitlines() if pattern.search(line))
        if stdout:
            stdout += "\n"
    output_path.write_text(stdout)
    if proc.stderr:
        print(proc.stderr)
    return get_header(argv)


def show_header(file, line1=30, line2=43):
    output_path = output_dir / file
    lines = output_path.read_text().splitlines()
    selected = "\n".join(lines[line1 - 1:line2])
    print(f"\nshowing {output_path}, lines {line1}-{line2}\n")
    print(selected)


In [ ]:
# To print the output 

def weighted_percentile(values, percentiles, weights=None, sw_sort=True, C= 0.5):                                                  
    """                                                                                                                            
    Get percentiles of unequal array
    Reference: 
    https://stackoverflow.com/questions/21844024/weighted-percentile-using-numpy
    See                                                                                                                            
    https://en.wikipedia.org/wiki/Percentile#The_weighted_percentile_method                                                        
    about the constant C that takes 0, 0.5 or 1                                                                                    
    """                                                                                                                            
    values = np.array(values)                                                                                                      
    percentiles = np.array(percentiles)                                                                                            
    if weights is None:                                                                                                            
        weights = np.ones(len(values))                                                                                             
    weights = np.array(weights)                                                                                                    
    assert np.all(percentiles >= 0) and np.all(percentiles <= 1), \
            'percentiles should be in [0, 1]'                                                                                      
    assert C==0 or C==0.5 or C==1, \
            'C should be 0, 0.5 or 1'                                                                                              
                                                                                                                                   
    if sw_sort:                                                                                                                    
        sorter = np.argsort(values)                                                                                                
        values = values[sorter]                                                                                                    
        weights = weights[sorter]                                                                                                  
                                                                                                                                   
    cumu_wts = np.cumsum(weights)                                                                                                  
    weighted_percentiles = (cumu_wts - C * weights) / (cumu_wts[-1] + (1-2*C)*weights)                                             
    return np.interp(percentiles, weighted_percentiles, values)                                                                    


def plot_hist_wt (axis, data, col = 'C1', log = 0, label = None, wt = None, NXBIN = 50) :
    xmin, xmax = weighted_percentile(data, [0.001, 0.999], weights=wt)
    if(log == 1) :
        xbins = np.logspace(np.log10(xmin), np.log10(xmax), num=NXBIN+1)
    else :
        xbins = np.linspace(xmin, xmax, num=NXBIN+1)
    y, x, pol = axis.hist(data, bins = xbins, histtype = 'step', linewidth = 2, color = col, label = label, weights = wt)
    xpers  = weighted_percentile(data, [0.1587, 0.50, 0.8413], weights=wt)
    # print (xpers)
    # xpers = np.percentile(data, q=100*np.array([0.1587, 0.50, 0.8413]))
    if log == 1 :
        xmids = 0.5 * (np.log10(xbins)[1:] + np.log10(xbins)[0:-1])
        dxpers = np.array([np.abs(xmids - np.log10(xper)) for xper in xpers])
        axis.set_xscale("log")
    else :
        xmids = 0.5 * (xbins[1:] + xbins[0:-1])
        dxpers = np.array([np.abs(xmids - xper) for xper in xpers])
    ix_pers = dxpers.argmin(1) 
    lss = ['--', '-', '--']
    for i, xeach in enumerate(xpers) :
        yeach = y[ix_pers[i]]
        axis.vlines(xeach, 0, yeach, ls = lss[i], color = col)
    return xpers
        
        
def print_data (file, header, DISP = True, usecols = None) :
    # Read the list
    data = np.loadtxt(file, usecols = usecols)
    if (DISP) :
        dataset = pd.DataFrame(data, columns=header)
        display(dataset)
        print("Unweighted statistics (just for reference) : ")
        display(dataset.describe())
    return data

def hist_wt (data, header, logparams = [], wtcol = None, PLOT = True, NXBIN = 50) :
    nparam = data.shape[1]
    dataT = data.T
    # Set wt array
    if wtcol is None :
        wt = np.ones(data.shape[0])
        wtcol = nparam + 1 # set non-exist column 
    else :
        wt =  dataT[wtcol]

    if PLOT:  # plot histogram
        if wtcol < nparam :
            print("Histogram weighted by ", header[wtcol], " : ")
        plt.rcParams["font.size"] = 13
        
        # Determine number of panels
        Nx = int(np.sqrt(nparam))
        if(Nx != np.sqrt(nparam)) : Nx += 1
        Ny = int(nparam / Nx)
        if(Nx * Ny < nparam) : Ny += 1

        fig, ax = plt.subplots(Ny, Nx, figsize=(14, 14*Ny/Nx))
        plt.subplots_adjust(wspace=0.4, hspace=0.4)


        for i in range(Ny*Nx) :
            # Skip wtcol param
            if i == wtcol : 
                continue

            # Get (ix, iy)
            if i < wtcol :
                ix = int(i/Nx)
                iy = i % Nx
            else :
                ix = int((i-1)/Nx)
                iy = (i-1) % Nx
                if i == Ny*Nx - 1 :
                    ax[int(i/Nx), i % Nx].axis('off') # deactivate the panel unused due to wtcol

            if i >= nparam :
                ax[ix,iy].axis('off')
                continue

            if (header[i] in logparams) :
                LOG = 1
            else :
                LOG = 0

            # print(header[i])
            plot_hist_wt(ax[ix, iy], dataT[i], col = 'C0', log = LOG, wt = wt, NXBIN = NXBIN)
            if iy == 0 : 
                ax[ix,iy].set_ylabel(r'Number', fontsize = 16)
            ax[ix,iy].set_xlabel('{}'.format(header[i]), fontsize = 16)
            # ax[ix,iy].legend(fontsize = 18, loc='upper left')


    # percentiles = np.array([weighted_percentile(dataT[i], [0.0228, 0.1587, 0.50, 0.8413, 0.9772], weights=wt) for i in range(nparam)])
    percentiles = np.zeros([nparam, 5])
    for i in range(nparam) :
        if i == wtcol : continue
        percentiles[i] = weighted_percentile(dataT[i], [0.0228, 0.1587, 0.50, 0.8413, 0.9772], weights=wt)
    
    return percentiles


## Sample 1. Posterior distribution of the lens mass and distance for a planetary event
In [Koshimoto et al. (2014)](https://ui.adsabs.harvard.edu/abs/2014ApJ...788..128K/abstract), $t_E = 34.0 \pm 2.2$ days and $\theta_E = 0.28 \pm 0.03$ mas are measured for OGLE-2008-BLG-355 occurred at $(l, b) = (-0.08, -3.45)$ deg.  
They also put a lower limit on the lens magnitude (upper limit on the lens brightness), $I_L > 20.67 \pm 0.47$ mag (this estimation of error looks suspicious to me now, eight years later, but let's respectfully use it as it was then...).  
Let's calculate the posterior distribution of the lens mass and distance for this event using `genulens` .   
To put the $I_L$ constraint[^1](#1), the extinction for the red clump, $A_I = 1.63$, is needed.  

<span id="1" style="font-size:small">1: 
Note on the $I_L$ constraint :  
The likelihood is calculated using the normal distribution with the magnitude unit, i.e., $\exp[-0.5(I_L - I_{L, obs})^2/I_{L, err}^2]$. This could be appropriate if the error is dominated by the calibration error.  If the error is dominated by the photon noise, it should be calculated in flux unit, which is not currently supported by `genulens`. Nevertheless, if the error in magnitude is small enough, it works fine because a linear approximation is applicable to the conversion between magnitude and flux.  In the current case of $20.67 \pm 0.47$ mag, the error is large, though. So, this is not appropriate unless the error is dominated by calibration error.
</span>


In [ ]:
# I can do the following by one line, like 
# argtxt1 = 'l -0.08 b -3.45 tE 34.0 2.2 thetaE 0.28 0.03 IL 20.67 0.47 ILdet 2 AIrc 1.63 VERBOSITY 3'
# But I divide it into some just to make each option clearer
argtxt1 = ''
argtxt1 += ' l -0.08 b -3.45' # coordinate
argtxt1 += ' tE 34.0 2.2'     #  tE constraint
argtxt1 += ' thetaE 0.28 0.03'  #  thetaE constraint
argtxt1 += ' IL 20.67 0.47 ILdet 2'  #  IL constraint, lower limit
argtxt1 += ' AIrc 1.63'  #  A_I value for red clump. Needed to calculate I_{L,0}
argtxt1 += ' VERBOSITY 3'  # 2 or 3. VERBOSITY controls output parameters

print("Sample 1 argument for genulens : ")
print(argtxt1)

file1 = 'sample1.dat' # output file 

### 1.1 Test run with small `NlikeMIN`
Before any possibly long calculation, I recommend doing a test run with a small `NlikeMIN` value to find out how long the calculation would take to have a desired resolution (say, $10^5$ accepted events).  
This time we can expect a relatively short calculation time because the number of observational constraints (i.e., detection of $t_E$ and $\theta_E$ and lower limit of $I_L$) are not many and the values are not uncommon.  
Nevertheless, I'll do a test run with `NlikeMIN` = 10000 for the sake of example.

In [ ]:
%%time
# Test run 
NlikeMIN = ' NlikeMIN 10000'
header = run_genulens(file1, argtxt1 + NlikeMIN)

This test run takes ~2 secs to have 10000 accepted events. (**Additional note on 2025/3/13: It took ~2 sec on my old laptop, whereas the current wall time is ~0.8 sec. All subsequent time descriptions refer to the results from my old laptop.**)  
Considering ~1 sec overhead for calculations other than the Monte Carlo simulation part, we can expect ~10 secs to have $10^5$ accepted events.  
Because this is acceptable time to me, I will try `NlikeMIN` = 1e+5 run next.  

At this time, it is also recommended to check the header part of the output file (i.e., sample1.dat in this case) to ensure that the intended values are correctly recognized;

In [ ]:
# Show line 36 - 53 of sample1.dat.
show_header(file1, 36, 53)

Note that since version 1.2 of `genulens`, only $t_E$ and $\theta_E$ values within the sampling parameter range are simulated for (much) faster simulation.  
When `p [val] [err]` is the constraint (p = tE, thetaE this case), the sampling range is set to be val-c\*err < p < val+c\*err, where c = 1.02 for UNIFORM = 1 while c = 4 for UNIFORM = 0 by default. You can change the constant c using `fIStE [c value]` or `fISthetaE [c value]` option, or omit this new feature using the option `NOIS 1`. (The same run takes ~7 sec with `NOIS 1` )  
In the current case, the sampling parameter ranges are 25.2000 < tE < 42.8000 and 0.1600 < thetaE < 0.4000 according to the header.  

### 1.2 Run with desired `NlikeMIN`

In [ ]:
%%time
# run to have the posterior distribution of ML and DL
NlikeMIN = ' NlikeMIN 1e+5'
header = run_genulens(file1, argtxt1 + NlikeMIN) # overwrite file1

This took ~15 secs, which is more or less the expected time.  
Then, let's plot the posterior and determine the lens mass and distance.

In [ ]:
wtcol = 0  # column of wtj, weight for each line
data = print_data(file1, header, DISP = True)
ptiles = hist_wt(data, header, logparams = ['M_L', 'pi_E'], wtcol = wtcol, PLOT = True) # logparams: header name you want to plot in logscale

and the percentiles of each parameter are

In [ ]:
print ("Percentiles weighted by ", header[wtcol] , " : ")
print ("#             -2sigma      -1sigma       median      +1sigma      +2sigma")
for i, ptile in enumerate(ptiles):
    if ptile[4] == ptile[0] : continue
    print('{:8s} {:12.5e} {:12.5e} {:12.5e} {:12.5e} {:12.5e}'.format(header[i], ptile[0], ptile[1], ptile[2], ptile[3], ptile[4]))

So, the median and 1 sigma values of the posterior distributions of M_L and D_L are $M_L = 0.44^{+0.31}_{-0.22} M_{\odot}$ and $D_L = 7.5^{+0.8}_{-1.0}$ kpc.  Note that the pi_EN and pi_EE distributions are not correctly calculated because we haven't inputted the projected Earth velocity this time.

In the above, each line needs to be weighted by wtj, but the statistics for the unweighted case were also displayed using pandas.DataFrame.describe for comparison.

## Sample 2. Prior probability of microlens parallax

Currently, the measurement of microlens parallax is susceptible to systematic errors in the light curve, and modeling often results in an unlikely large value (e.g., $\pi_{\rm E} > 1$) and/or a direction of microlens parallax vector rarely expected considering the Galactic rotation.  
Even when a systematic error is not a problem, it is often the case only one-dimensional microlens parallax is measured.  
In both cases, a prior probability distribution of $(\pi_{\rm E,N}, \pi_{\rm E,E})$ based on a Galactic model helps your modeling.  
Let's see how this can be calculated with `genulens`, using the actual calculations I did in the analysis of OGLE-2018-BLG-1185 as an example.

In the OGLE-2018-BLG-1185 analysis ([Kondo et al. 2021](https://ui.adsabs.harvard.edu/abs/2021AJ....162...77K/abstract)), we have 1-D parallax measurement by the Spitzer telescope.  
We had tentatively measured $t_{\rm E} = 16.0 \pm 0.14$ days and $\theta_{\rm E} = 0.207 \pm 0.015$ mas without using the Spitzer data.  
So, a prior probability can be calculated just by `genulens tE 16.0 0.14 thetaE 0.207 0.015 [other options]`.  
However, considering possible correlations between the $(\pi_{\rm E,N}, \pi_{\rm E,E})$ and other parameters (i.e., $t_E$ and $\theta_E$), we wanted to avoid applying the normal probability likelihood which is assumed by running the above command.  
So, we decided to use the `UNIFORM 1` option to apply an uniform probability likelihood with larger uncertainties, which would allow us to multiply it by the probability distribution coming from the MCMC stationary distribution later.  (see Section 1.2 of [Usage.pdf](https://github.com/nkoshimoto/genulens/blob/main/Usage.pdf) for detail about this idea )

In [ ]:
argtxt2 = ''
argtxt2 += ' l 2.47 b -2.00' # coordinate
argtxt2 += ' vEarthlb -24.3835 13.0860' # Earth projected velocity (l, b) in km/sec at t0. Needed to calculate (pi_E,N, pi_E,E)

# tE and thetaE constraints with significantly increased errors so that it covers major parameter space explored by MCMC. 
tEval, tEerr = 16.0, 0.5
thetaEval, thetaEerr = 0.207, 0.050
argtxt2 += ' tE {:.2f} {:.2f}'.format(tEval,tEerr)     #  tE constraint. 
argtxt2 += ' thetaE {:.3f} {:.3f}'.format(thetaEval, thetaEerr)  #  thetaE constraint.
argtxt2 += ' UNIFORM 1'  # use uniform probability as likelihood (eq. 2 in Section 1.1 of Usage.pdf)

# For D_S distribution (see Section 3-I-3.1 in Usage.pdf)
argtxt2 += ' Isrange 19.8 20.4 AIrc 1.96' #  Needed to calculate I_{S,0} as a function of D_S
argtxt2 += ' VIsrange 2.24 2.44 EVIrc 1.66' #  Needed to calculate (V-I)_{S,0} as a function of D_S
argtxt2 += ' DMrc 14.47' # mean RC distance modulus taken from the OGLE extinction calculator (Nataf+13)

# To output events with small probability 
argtxt2 += ' SMALLGAMMA 1' # see Section 3-II-2 in Usage.pdf

# output parameters
argtxt2 += ' VERBOSITY 3'  # 2 or 3.  I used 2 in the actual calculation to reduce the output size, but I use 3 for here

print("Sample 2 argument for genulens : ")
print(argtxt2)

file2 = 'sample2.dat'

Difference from Sample 1 is :
- `vEarthlb -24.3835 13.0860` :   
Earth projected velocity (l, b) in km/sec at t0. Needed to calculate $(\pi_{\rm E,N}, \pi_{\rm E,E})$ correctly.


- `tE 16.00 0.50 thetaE 0.207 0.050 UNIFORM 1` :   
The `UNIFORM 1` option uses the uniform probability 
$$
{\cal L}_{p} = 
\begin{cases}
1 & \text{ when $|\texttt{val} - p| \leq \texttt{err}$} \\
0 & \text{  when $|\texttt{val} - p| > \texttt{err}$ } 
\end{cases}
$$  
as likelihood instead of $\exp[-0.5(p - \texttt{val})^2/\texttt{err}^2]$ when `p [val] [err]` is given. (p = tE, thetaE, here).  
Accordingly, the errors for tE and thetaE are taken significantly larger than $1 \sigma$ so that it covers major parameter space explored by MCMC.  
Again see Section 1.2 of [Usage.pdf](https://github.com/nkoshimoto/genulens/blob/main/Usage.pdf) for the detail of the idea behind this.


- `Isrange 19.8 20.4 AIrc 1.96 VIsrange 2.24 2.44 EVIrc 1.66 DMrc 14.47` :   
These are needed to calculate the $D_S$ distribution with the source mag constraints.  Narrowing the allowed value of $D_S$ can also narrow the allowed range of parallax vector. How the $D_S$ distribution is calculated is presented in Section 3-I-3.1 of [Usage.pdf](https://github.com/nkoshimoto/genulens/blob/main/Usage.pdf). Note that adding these does not slow down the calculation since the calculation of the $D_S$ distribution is done before the Monte Carlo simulation part.


- `SMALLGAMMA 1` :   
Outputs all generated events although `genulens`  normally does not output events with extremely small event rate ($\propto D_L^2 \theta_{\rm E} \mu_{\rm rel}$) to reduce the amount of output.  See Section 3-II-2 of [Usage.pdf](https://github.com/nkoshimoto/genulens/blob/main/Usage.pdf) for detail of this option.  
This time I use this option to quantitatively calculate the probability distribution of $(\pi_{\rm E,N}, \pi_{\rm E,E})$ over a wide area of the parameter space, including very low probability parts. 


### 2.1 Test run

So, let's do a test run. (with a larger number because acceptance ratio with `SMALLGAMMA 1` is larger than without it.)

In [ ]:
%%time
# Test run 
NlikeMIN = ' NlikeMIN 1e+5'
header = run_genulens(file2, argtxt2 + NlikeMIN)

In [ ]:
show_header(file2, 30, 48)

### 2.2 Parallel run (not executed here)

Now that we aim to have a moderate resolution in the 2D probability distribution of $(\pi_{\rm E,N}, \pi_{\rm E,E})$ including low probability areas of the parameter space, we need much larger simulation than the previous case.  
There was no clear rationale, but we decided to output 50 million events. I could have done the calculation on my laptop if I had taken the time, but fortunately, I was allowed to use a server at Osaka University, so I did the calculations in parallel.  
In this notebook, parallel computation is not performed, but there is a caveat that the random seed value must be changed in the argument for parallel computation, so I only show an example of the execution commands below, and then actually run just one of them.  

Although now the above test only took ~2 secs thanks to the importance sampling feature introduced in version 1.2, the same command had taken ~30 secs until version 1.1 of `genulens`.
So when I analyzed this event, I decided to divide the 50 million calculations into 50 parallel runs with `NlikeMIN 1e+6` ;

In [ ]:
# Just to show the commands for the parallel run.
Nrun = 50
NlikeMIN = ' NlikeMIN 1e+6'
commands = [argtxt2 + NlikeMIN + ' seed {}'.format(i+1) for i in range(Nrun)]
print("# Example commands for a parallel run : \n")
for command in commands :
    print(GENULENS_EXE, command)

Note that the seed value must be a positive integer. 

### 2.3 Run one of the commands in the parallel computation
Here, just run one of them.  
(Note that the following command will create a 160 MB file in your directory, so be careful if your storage space is limited. You can delete the file later of course.)

In [ ]:
%%time
# Run one command
header = run_genulens(file2, commands[0])

In [ ]:
wtcol = 0  # column of wtj, weight for each line
data = print_data(file2, header, DISP = True)
ptiles = hist_wt(data, header, logparams = ['M_L', 'pi_E'], wtcol = wtcol, PLOT = True) # logparams: header name you want to plot in logscale

In [ ]:
print ("Percentiles weighted by ", header[wtcol] , " : ")
print ("#             -2sigma      -1sigma       median      +1sigma      +2sigma")
for i, ptile in enumerate(ptiles):
    if ptile[4] == ptile[0] : continue
    print('{:8s} {:12.5e} {:12.5e} {:12.5e} {:12.5e} {:12.5e}'.format(header[i], ptile[0], ptile[1], ptile[2], ptile[3], ptile[4]))

The $(\pi_{\rm E,N}, \pi_{\rm E,E})$ distribution looks like 

In [ ]:
# Plot P(pi_EN, pi_EE)
dataT = data.T
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plt.rcParams["font.size"] = 22
# ax = fig.add_subplot(111)
NXBIN, NYBIN = 100, 200
xmin, xmax = -0.7, 0.7
ymin, ymax = -1.4, 1.4
# ZMIN, ZMAX
xedges =     np.linspace(xmin, xmax, NXBIN+1)
yedges =     np.linspace(ymin, ymax, NYBIN+1)
H2D, xedges, yedges = np.histogram2d(dataT[8], dataT[7], weights= dataT[0], bins=(xedges, yedges))
import matplotlib as mpl
plt.pcolormesh(xedges, yedges, H2D.T, cmap='jet', norm=mpl.colors.LogNorm())
cbar= plt.colorbar()
ax.set_xlabel(r'$\pi_{\rm E, E}$')
ax.set_ylabel(r'$\pi_{\rm E, N}$')
ax.set_aspect("equal", "box")
ax.invert_xaxis()

See Fig. 5 of [Kondo et al. (2021)](https://ui.adsabs.harvard.edu/abs/2021AJ....162...77K/abstract) for a higher resolution version of the distribution using all 50 million events.

## Sample 3. Planetary event analysis with a white dwarf host possibility
The above analysis assumed that the lens objects were ordinary stars, but the discovery of a planet around a white dwarf by [Blackman et al. (2021)](https://ui.adsabs.harvard.edu/abs/2021Natur.598..272B/abstract) implies that white dwarfs also have planets with non-negligible frequency in the microlensing sensitive region.  
You can add the white dwarf lens possibility using the `onlyWD 1` option. Let's repeat the sample 1 analysis with that option. 

In [ ]:
argtxt3 = argtxt1 + ' onlyWD 1'
file3 = 'sample3.dat'

In [ ]:
%%time
# run to have the posterior distribution of ML and DL
NlikeMIN = ' NlikeMIN 1e+5'
header = run_genulens(file3, argtxt3 + NlikeMIN) 

In [ ]:
wtcol = 0  # column of wtj, weight for each line
data = print_data(file3, header, DISP = True)
ptiles = hist_wt(data, header, logparams = ['M_L', 'pi_E'], wtcol = wtcol, PLOT = True) # logparams: header name you want to plot in logscale

You can see that a sharp peak has appeared at ~0.5 Msun in the M_L distribution, corresponding to the white dwarf population.    
Also, the fREM now takes not only 0, but also 1, which indicates a white dwarf lens.  
Finally, note that using the `onlyWD 1` option for a planetary event analysis implicitly assumes that the planet-hosting probability is the same for a white dwarf as for a normal star.

## Sample 4. Black hole candidate analysis
You can consider a neutron star or black hole lens possibility by using the `REMNANT 1` option.  
Let's apply that option to a Bayesian analysis for a black-hole candidate OGLE3-ULENS-PAR-02 ([Wyrzykowski et al. 2016](https://ui.adsabs.harvard.edu/abs/2016MNRAS.458.3012W/abstract)).
 

In [ ]:
# u0-, piEN- solution
argtxt4 = ''
argtxt4 += ' l 1.45 b -2.13' # coordinate
argtxt4 += ' vEarthlb -13.24 19.67' # Earth projected velocity (l, b) in km/sec at t0. Needed to calculate (pi_E,N, pi_E,E)

# tE and piE constraints. 
tEpiEobs = ''
tEpiEobs += ' tE 288.8 8.0'     #  tE constraint. 
tEpiEobs += ' piEN -0.033 0.001 piEE -0.073 0.004'  #  parallax constraint.  

# For D_S distribution 
argtxt4 += ' Isrange 15.30 15.60 AIrc 1.42' #  taken conservatively large
argtxt4 += ' VIsrange 1.89 2.54 EVIrc 1.14' #  roughly read from the CMD 
argtxt4 += ' DMrc 14.494' # mean RC distance modulus taken from the OGLE extinction calculator (Nataf+13)

# IL constraint 
argtxt4 += ' IL 17.7 0.01 ILdet 2' # lower limit on IL

# To output events with small probability.
argtxt4 += ' SMALLGAMMA 1' # see Section 3-II-2 in Usage.pdf

# To consider the REMNANT possibility
argtxt4 += ' REMNANT 1'

# output parameters
argtxt4 += ' VERBOSITY 3'  # 2 or 3.  

print("Sample 4 argument for genulens : ")
print(argtxt4 + tEpiEobs)

file4 = 'sample4.dat'

### 4.1 Test run (comparison between parallax vector and amplitude constraints)
This time the computation is expected to be very slow because of  
1. An uncommonly large tE value :  
In each trial of the Monte Carlo simulation, `genulens` calculates $M_{\rm cand} = (\mu_{\rm rel} t_{\rm E, cand})^2/(\kappa \pi_{\rm rel})$ from a randomly generated combination of ($D_L$, $D_S$, $\mu_{\rm rel}$) and the candidate $t_{\rm E}$ value determined from the input $t_E$ range. However, if the input $t_E$ value is large, this $M_{\rm cand} \propto t_{\rm E, cand}^2$ is quite likely to be larger than the maximum mass of the mass function, so many simulations are needed to have a reasonable mass distribution. 


2. ($\pi_{\rm E, N}, \pi_{\rm E, E}$) constraints with small error bars :  
Although `genulens` is good at simulation limited to the range of the input $\pi_E$, it is not able to limit simulation to a certain sky direction. So it is time-consuming if the parallax vector direction is specified, especially when the error bars are small.

These are two features that `genulens` is currently not very good at, and we want to avoid them if possible.  
Although the large tE value is inevitable, parallax vector constraints might be avoidable if there is no essential difference from the results with parallax amplitude constraint.  
Here, let's see if there is a significant difference between them with a small NlikeMIN value.  


### 4.1.1  With parallax vector (piEN, piEE) constraint


In [ ]:
# tE and piE constraints. 
tEpiEobs = ''
tEpiEobs += ' tE 288.8 8.0'     #  tE constraint. 
tEpiEobs += ' piEN -0.033 0.001 piEE -0.073 0.004'  #  parallax constraint.  

In [ ]:
%%time
# Test run for sample 4 with piEN, piEE
NlikeMIN = ' NlikeMIN 500'
header = run_genulens(file4, argtxt4 + tEpiEobs + NlikeMIN)

With the real measured value, calculation took ~10 min only for a `NlikeMIN 500` calculation. The distribution with this looks like 

In [ ]:
wtcol = 0  # column of wtj, weight for each line
data = print_data(file4, header, DISP = False)
print ('Results of test run with', tEpiEobs)
ptiles = hist_wt(data[:, 0:9], header, logparams = ['M_L', 'theta_E', 'pi_E', 'mu_rel', 'aL', 'aLpmin'], wtcol = wtcol, PLOT = True, NXBIN = 50) # logparams: header name you want to plot in logscale

### 4.1.2 With parallax amplitude (piE) constraint

In [ ]:
# tE and piE constraints. 
tEpiEobs = ''
tEpiEobs += ' tE 288.8 8.0'     #  tE constraint. error bar doubled
tEpiEobs += ' piE {:.3f} 0.005'.format(np.sqrt(0.033**2 + 0.073**2))

In [ ]:
%%time
# Test run for sample 4 with 1D piE
NlikeMIN = ' NlikeMIN 500'
header = run_genulens(file4, argtxt4 + tEpiEobs + NlikeMIN)

In [ ]:
wtcol = 0  # column of wtj, weight for each line
data = print_data(file4, header, DISP = False)
print ('Results of test run with', tEpiEobs)
ptiles = hist_wt(data[:, 0:9], header, logparams = ['M_L', 'theta_E', 'pi_E', 'mu_rel', 'aL', 'aLpmin'], wtcol = wtcol, PLOT = True, NXBIN = 50) # logparams: header name you want to plot in logscale

So, the calculation with the parallax amplitude constraint is ~150 times faster than the calculation with the parallax vector constraint.  
These two results are similar, and both have bimodal M_L distributions: less massive lens with M_L < ~1.3 Msun and black hole lens with M_L > 5 Msun. Therefore, if this is a calculation for publication, I would calculate the 2-D parallax one using a parallel computation, but here we will calculate the 1-D parallax one, to save our time.

### 4.2 Run with desired `NlikeMIN`

Even with the parallax amplitude constraint, the calculation is much slower than other sample calculations. So, here I use NlikeMIN = 50000.

In [ ]:
%%time
# run to have the posterior distribution of ML and DL
NlikeMIN = ' NlikeMIN 50000'
header = run_genulens(file4, argtxt4 + tEpiEobs + NlikeMIN)

In [ ]:
wtcol = 0  # column of wtj, weight for each line
data = print_data(file4, header, DISP = True)
ptiles = hist_wt(data, header, logparams = ['M_L', 'theta_E', 'pi_E', 'mu_rel', 'aL', 'aLpmin'], wtcol = wtcol, PLOT = True, NXBIN = 50) # logparams: header name you want to plot in logscale

In [ ]:
print ("Percentiles weighted by ", header[wtcol] , " : ")
print ("#             -2sigma      -1sigma       median      +1sigma      +2sigma")
for i, ptile in enumerate(ptiles):
    if ptile[4] == ptile[0] : continue
    print('{:8s} {:12.5e} {:12.5e} {:12.5e} {:12.5e} {:12.5e}'.format(header[i], ptile[0], ptile[1], ptile[2], ptile[3], ptile[4]))

The flag fREM = 2 indicates a neutron star and fREM = 3 indicates a black hole.
Note that results on a black hole candidate event highly depends on the mass function used. `genulens` calculates the present-day mass function by combining the initial mass function by [Koshimoto, Baba, and Bennett (2021)](https://ui.adsabs.harvard.edu/abs/2021ApJ...917...78K/abstract) with the initial-final mass relations for compact objects by [Lam et al. (2020)](https://ui.adsabs.harvard.edu/abs/2020ApJ...889...31L/abstract).

## Sample 5. Influence of Black Hole Kick Velocity on Microlensing Distributions (added on Mar 2025)
In [Koshimoto, Kawanaka, and Tsuna (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...973....5K/abstract), we considered the effect of inflating the scale height of the BH population as a function of the BH kick velocity on microlensing distribution.  

In Mar 2025, I implemented the function used for the above study into genulens.

I reproduce here the $v_{\rm avg} = 400~{\rm km/s}$ panel of Fig. 5 and the $v_{\rm avg} = 25~{\rm km/s}$ panel of Fig. 6 of [Koshimoto, Kawanaka, and Tsuna (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...973....5K/abstract) as an example of the new options.  

For details of the model, please refer to the paper.

### 5.1 The $v_{\rm avg} = 400~{\rm km/s}$ panel of Fig. 4 of [Koshimoto, Kawanaka, and Tsuna (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...973....5K/abstract)

In [ ]:
# Common arguments  
argtxt5 = ''
argtxt5 += ' l -0.14 b -1.62' # coordinate of OGLE-2011-BLG-0462
argtxt5 += ' REMNANT 1' # to simulate BHs
argtxt5 += ' VERBOSITY 3' # control output parameters

# No change in scale height or surface density (same treatment as Koshimoto+21)
argtxt5_0 = argtxt5 + ' vkickBH 400 MXDkick 1 BHhd 0 UseSigBH 0'

# With change in scale height and surface density
argtxt5_1 = argtxt5 + ' vkickBH 400 MXDkick 1 BHhd 1 UseSigBH 1'

print("Sample 5.1 arguments for genulens : ")
print(argtxt5_0)
print(argtxt5_1)

file5_0 = 'sample5_0.dat'
file5_1 = 'sample5_1.dat'

The newly added options are :
- `vkickBH [double]` (default: 100) :
BH kick velocity value in km/sec (see the `MXDkick` description below for what this value is).  


- `MXDkick [0 or 1]` (default: 1) :   
When 1, use Maxwell distribution with the average velocity of $v_{\rm avg} = $ `vkickBH`.  
When 0, use constant kick velocity of $v_{\rm avg} = $ `vkickBH`.  


- `BHhd [0 or 1]` (default: 1):   
When 1, change BH disk scale height following Eq. (5) of [Koshimoto, Kawanaka, and Tsuna (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...973....5K/abstract).  
When 0, use the stellar disk scale height also for BHs.  


- `UseSigBH [0 or 1]` (default: 1):   
When 1, change BH disk surface density following Eq. (10) of [Koshimoto, Kawanaka, and Tsuna (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...973....5K/abstract).  
When 0, use the stellar disk surface density also for BHs.  


Note that the modification of the surface density by `UseSigBH` depends on the value of `vkickBH`. However, it is designed to work properly only for the specific values of vkickBH = 25, 50, 100, 200, or 400.  For other values, the surface density adjustment corresponding to the closest one among these five (on a logarithmic scale) is applied.


To make BH population enough to see its distribution, I simulate Nsimu = $3 \times 10^7$ events.  
I also use 'grep 3$' to save the output file size. (BH events have fREM flag = 3, which outputted in the far right with VERBOSITY 3 option.)

In [ ]:
%%time
# run to output BH events
Nsimu = ' Nsimu 3e+7'
header = run_genulens(file5_0, argtxt5_0 + Nsimu, grep = ' 3$')
header = run_genulens(file5_1, argtxt5_1 + Nsimu, grep = ' 3$')

In [ ]:
wts0, tEs0, iLs0, fREMs0 = np.loadtxt(file5_0, usecols=[0, 4, 15, 16], unpack=True)
wts1, tEs1, iLs1, fREMs1 = np.loadtxt(file5_1, usecols=[0, 4, 15, 16], unpack=True)

In [ ]:
Nsimu = 3e+7
fig = plt.figure(figsize = (4, 4))
plt.rcParams["font.size"] = 17

xmin, xmax = 4, 1300
ymin, ymax = 0, 4.5e-04
ax1 = fig.add_subplot(111)
tEBH = tEs0[fREMs0 == 3]
wtBH = wts0[fREMs0 == 3]
iLBH = iLs0[fREMs0 == 3] 
col = 'C0'
ll, tEmed0, ul = plot_hist_wt(ax1, tEBH[iLBH < 8], wt = wtBH[iLBH < 8]/Nsimu, log=1,col = col, label = r'No inflation (K21)')
fBHsum = np.sum(wtBH[iLBH < 8]/Nsimu)
ax1.text(xmax*(xmin/xmax)**0.01, ymax - 0.08*(ymax-ymin), r'$f_{\rm BH, disk}=$' + '{:.4f}'.format(fBHsum), 
         fontsize = 18, color= col, horizontalalignment='right')

tEBH = tEs1[fREMs1 == 3]
wtBH = wts1[fREMs1 == 3]
iLBH = iLs1[fREMs1 == 3]
col = 'C1'
ll, tEmed1, ul = plot_hist_wt(ax1, tEBH[iLBH < 8], wt = wtBH[iLBH < 8]/Nsimu, log=1,col = col, label = r'Disk inflation (K24)')
fBHsum = np.sum(wtBH[iLBH < 8]/Nsimu)
ax1.text(xmax*(xmin/xmax)**0.01, ymax - 0.18*(ymax-ymin), r'$f_{\rm BH, disk}=$' + '{:.4f}'.format(fBHsum), 
         fontsize = 18, color= col, horizontalalignment='right')
ax1.text(xmax*(xmin/xmax)**0.01, ymax - 0.29*(ymax-ymin), r'<$t_{\rm E}$>'+'= {:.0f} d'.format(tEmed1), 
         fontsize = 18, color= col, horizontalalignment='right')
ax1.set_xlim(xmin, xmax)
ax1.set_ylim(ymin, ymax)
ax1.set_xlabel(r'$t_E$ (days)', fontsize = 20)
ax1.set_title(r'Reproduction of the $v_{\rm avg}=$' + '400 km/s panel of Fig. 4 of Koshimoto+24', fontsize = 17)
ax1.set_ylabel(r'Event fraction', fontsize = 20)
ax1.legend(fontsize = 20, loc='upper left', bbox_to_anchor=(1.05, 1.00), borderaxespad=0)

fig.subplots_adjust(left=0.05, right=0.95, bottom=0.1, top=0.9, hspace = 0.25, wspace = 0.3)


### 5.2 The $v_{\rm avg} = 25~{\rm km/s}$ panel of Fig. 6 of [Koshimoto, Kawanaka, and Tsuna (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...973....5K/abstract)

In [ ]:
# Set v_avg = 25 km/s and add NoGAMMAIS option
argtxt5_2 = argtxt5 + ' vkickBH 25 NoGAMMAIS 1'

print("Sample 5.2 arguments for genulens : ")
print(argtxt5_2)

file5_2 = 'sample5_2.dat'

This time I intentionally omitted MXDkick, BHhd, and UseSigBH from the input arguments because their default value is 1.  
I added the `NoGAMMAIS 1` option to make wtj closer to 1 since plt.scatter does not accept an weight list.

In [ ]:
%%time
# run to output BH events
Nsimu = ' Nsimu 2e+5'
header = run_genulens(file5_2, argtxt5_2 + Nsimu)

In [ ]:
tEs, piEs, fREMs = np.loadtxt(file5_2, usecols=[4, 6, 16], unpack=True)

In [ ]:
fig = plt.figure(figsize = (4, 4))
cols = ['C0', 'C4', 'C1', 'black']
labels = ['star', 'WD', 'NS', 'BH']
tEobs = 270.7
tEe = 11.2
piEobs = 0.0894
piEe = 0.0135
l_tick_mi = 4
l_tick_ma = 8

ax1 = fig.add_subplot(111)
for iRem in range(4):
    indices = np.where(fREMs == iRem)
    ax1.scatter(tEs[indices], piEs[indices], c = cols[iRem], s = 8, label = labels[iRem])
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r'$t_E$ (days)', fontsize = 19)
ax1.set_ylabel(r'$\pi_E$', fontsize = 19)
ax1.set_xlim(0.1, 1200)
ax1.set_ylim(0.001, 1.0)
ax1.set_title(r'Reproduction of the $v_{\rm avg}=$' + '25 km/s panel of Fig. 6 of Koshimoto+24', fontsize = 17)
ax1.errorbar(tEobs, piEobs, xerr= tEe, yerr= piEe, 
                    capsize= 7, fmt='o', markersize=7, ecolor='red', markeredgecolor = "red", color='w', label = 'OB110462')
ax1.legend(fontsize = 21, loc='upper left', bbox_to_anchor=(1.05, 1.00), borderaxespad=0)
# Lam+20's threshold of (tE, piE) = (120, 0.08)
ax1.vlines(120, 0, 0.08, ls = '--', color = 'cyan')
ax1.hlines(0.08, 120, 10000, ls = '--', color = 'cyan')

fig.subplots_adjust(left=0.05, right=0.95, bottom=0.1, top=0.9, hspace = 0.3, wspace = 0.3)

## Remove the output files if you want

In [ ]:
# Do not do these cells if you want to keep sample1.dat, sample2.dat, sample3.dat, or sample4.dat
# Remove sample1.dat
try: 
    os.remove(file1)
except FileNotFoundError: 
    pass

In [ ]:
# Remove sample2.dat
try: 
    os.remove(file2)
except FileNotFoundError: 
    pass

In [ ]:
# Remove sample3.dat
try: 
    os.remove(file3)
except FileNotFoundError: 
    pass

In [ ]:
# Remove sample4.dat
try: 
    os.remove(file4)
except FileNotFoundError: 
    pass

In [ ]:
# Remove files made in Sample 5
files = [file5_0, file5_1, file5_2]
for file in files:
    try: 
        os.remove(file)
    except FileNotFoundError: 
        pass